### LOAD EVERYTHING

In [6]:
import pandas as pd
import pickle

# Load data
data = pd.read_csv("cleaned_medicines.csv")

with open("medicine_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

### Recreate search logic

In [11]:
from rapidfuzz import process, fuzz

def search(query, k=10):
    query = query.lower().strip()

    # 🔥 Step 1: Fuzzy match top candidates (FAST filter)
    names = data['name'].astype(str).tolist()
    fuzzy_matches = process.extract(query, names, limit=50)

    candidate_indices = [match[2] for match in fuzzy_matches]
    candidates = data.iloc[candidate_indices]

    # 🔥 Step 2: TF-IDF on reduced set
    query_vector = vectorizer.transform([query])
    candidate_vectors = vectorizer.transform(candidates['search_field'])

    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity(query_vector, candidate_vectors)[0]

    # 🔥 Step 3: Combine fuzzy + semantic score
    combined = []
    for i, (_, score, _) in enumerate(fuzzy_matches):
        tfidf_score = sims[i]
        fuzzy_score = score / 100

        final_score = 0.6 * tfidf_score + 0.4 * fuzzy_score

        combined.append((final_score, candidate_indices[i]))

    # Sort and pick top-k
    combined.sort(reverse=True)

    top_indices = [idx for _, idx in combined[:k]]
    results = data.iloc[top_indices].copy()

    return results

### Define relevance

In [12]:
from rapidfuzz import fuzz

def is_relevant(query, row):
    query = query.lower()
    name = str(row["name"]).lower()
    symptoms = str(row["symptoms"]).lower()

    # fuzzy match for misspellings
    if len(query) > 4 and fuzz.partial_ratio(query, name) > 85:
        return True

    # token match for symptoms
    q_tokens = set(query.split())
    symp_tokens = set(symptoms.split())

    return len(q_tokens & symp_tokens) > 0

### Evaluation logic

In [13]:
import time
import numpy as np

def evaluate(queries):
    precisions, recalls, f1s, topk = [], [], [], []
    times = []

    for q in queries:
        start = time.time()
        results = search(q)  # 🔥 YOUR ACTUAL SYSTEM
        times.append(time.time() - start)

        tp = sum(1 for _, r in results.iterrows() if is_relevant(q, r))
        fp = 10 - tp

        # 🔥 FIXED recall logic
        total_relevant = 10

        fn = max(total_relevant - tp, 0)

        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
        topk.append(1 if tp > 0 else 0)

    return {
        "Precision": round(np.mean(precisions), 2),
        "Recall": round(np.mean(recalls), 2),
        "F1": round(np.mean(f1s), 2),
        "Top-10 Accuracy": round(np.mean(topk), 2),
        "Avg Time": round(np.mean(times), 3)
    }

### queries

In [14]:
exact = [
    "paracetamol", "ibuprofen", "aspirin", "amoxicillin", "azithromycin",
    "cetirizine", "loratadine", "metformin", "omeprazole", "pantoprazole",
    "diclofenac", "ranitidine", "doxycycline", "clindamycin", "ciprofloxacin",
    "levocetirizine", "insulin", "glimepiride", "atorvastatin", "amlodipine",
    "losartan", "atenolol", "telmisartan", "rabeprazole", "esomeprazole",
    "naproxen", "ketorolac", "tramadol", "codeine", "montelukast",
    "salbutamol", "budesonide", "fluticasone", "prednisolone", "hydrocortisone",
    "warfarin", "heparin", "clopidogrel", "rosuvastatin", "simvastatin",
    "glucophage", "augmentin", "crocin", "calpol", "combiflam",
    "vicks", "benadryl", "digene", "gelusil", "disprin"
]
misspelled = [
    "paracetmol", "ibrufen", "asprin", "amoxcillin", "azitromycin",
    "cetirizin", "loratidine", "metphormin", "omeprazol", "pantaprazole",
    "diclofinac", "ranitidin", "doxycicline", "clindamycine", "ciprofloxin",
    "levocetrizine", "insullin", "glimiperide", "atorvasttin", "amlodipin",
    "losartan", "atenolol", "telmisartan", "rabiprazole", "esomeprazol",
    "naproxin", "ketorolak", "tramdol", "codine", "montelucast"
]
symptoms = [
    "fever", "headache", "pain", "inflammation", "infection",
    "bacterial infection", "allergy", "sneezing", "runny nose",
    "itching", "hives", "diabetes", "blood sugar", "acidity",
    "gastric problem", "heart attack", "blood clot", "cough",
    "cold", "body pain", "joint pain", "stomach pain",
    "burning sensation", "indigestion", "gas", "skin allergy",
    "respiratory infection", "throat infection", "ear infection",
    "sinus infection", "high blood pressure", "cholesterol",
    "asthma", "bronchitis", "flu", "migraine",
    "muscle pain", "toothache", "back pain", "arthritis"
]

print("\n📊 FINAL RESULTS\n")

print("Exact:", evaluate(exact))
print("Misspelled:", evaluate(misspelled))
print("Symptom:", evaluate(symptoms))


📊 FINAL RESULTS

Exact: {'Precision': 0.7, 'Recall': 0.7, 'F1': 0.7, 'Top-10 Accuracy': 0.92, 'Avg Time': 1.945}
Misspelled: {'Precision': 0.73, 'Recall': 0.73, 'F1': 0.73, 'Top-10 Accuracy': 1.0, 'Avg Time': 2.339}
Symptom: {'Precision': 0.15, 'Recall': 0.15, 'F1': 0.15, 'Top-10 Accuracy': 0.35, 'Avg Time': 1.717}
